In [ ]:
import csv
import pandas as pd
import re
import os
from bs4 import BeautifulSoup
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import login
from dotenv import load_dotenv
import yfinance as yf
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt



load_dotenv()
token = os.getenv('HF_TOKEN')
login(token)


In [ ]:
News_api_data = pd.read_csv('NewsAPI_data.csv',encoding='utf-8')
guardian_data = pd.read_csv('guardian_data.csv',encoding='utf-8')
guardian_data = guardian_data.rename(columns={'headline': 'title', 'body_text': 'content'}, inplace=True)
news = pd.concat([guardian_data,News_api_data])
news['date'] = pd.to_datetime(news['date'], format='%Y%m%d')


In [ ]:
prosus_model= AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
prosus_tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')


nlp1 = pipeline('text-classification', model=prosus_model, tokenizer=prosus_tokenizer, truncation=True, max_length=512,batch_size=32)

def sentiment(texts):
    results1 = nlp1([i if isinstance(i, str) and i.strip() else ' ' for i in texts])
    return [ i['score'] if i['label'] == 'positive' else -i['score'] if i['label'] == 'negative' else 0 for i in results1] 

news['title_score_prosus'] = sentiment(news['title'])
news['content_score_prosus'] = sentiment(news['content'])
news['final_score_prosus'] = news[['title_score_prosus','content_score_prosus']].mean(axis=1)

In [ ]:
news.to_csv('important_one.csv', index=False, encoding='utf-8')

In [ ]:
def get_finance_data(ticker):
    pre_data = yf.download(ticker, start='2026-02-01', end=datetime.today().date())
    pre_data.reset_index(inplace=True)
    pre_data['daily_avg'] = pre_data[['Open','Close']].mean(axis=1)
    data = pre_data[['Date','daily_avg']]
    data.columns = ['date','daily_avg']
    data['date'] = pd.to_datetime(data['date'], format='%Y%m%d')


    return data

brent = get_finance_data('BZ=F')
wti = get_finance_data('CL=F')
OVX = get_finance_data('^OVX')

In [ ]:
print(news['date'].dtype)
print(brent['date'].dtype)
print(wti['date'].dtype)
print(OVX['date'].dtype)

In [ ]:
sentiment_by_day = pd.DataFrame(news.groupby('date')['final_score_prosus'].agg(['mean','std','count']))
sentiment_by_day.reset_index(inplace=True)

In [ ]:
pre_complete1 = pd.merge(sentiment_by_day, brent ,on='date', how='inner')
pre_complete2 = pd.merge(pre_complete1, wti,on='date', how='inner')
complete = pd.merge(pre_complete2, OVX,on='date', how='inner')
complete = complete.rename(columns={'mean': 'sentiment_mean','std': 'sentiment_std','daily_avg_x': 'brent','daily_avg_y': 'wti','daily_avg': 'ovx'})

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.bar(complete['date'], complete['sentiment_mean'], color='#2D244D', alpha=0.9, width=0.8, label='Daily Sentiment')
ax1.axhline(0, color='black')
ax1.set_ylabel('Sentiment Score by day\n(Lower is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')
ax1.set_ylim(-0.7, 0.2)

ax2 = ax1.twinx()
ax2.plot(complete['date'], complete['brent'], color='#FF5B04', linewidth=2.5, label='Brent Crude (USD)')
ax2.plot(complete['date'], complete['wti'], color='#C2441C', linewidth=2.5, label='WTI Crude (USD)')
ax2.set_ylabel('Oil Price (USD/barrel)', fontsize=11, fontfamily='monospace')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('News Sentiment vs Brent Crude Price Over the Iran War', fontsize=15, fontweight='bold', fontfamily='monospace')
plt.savefig('sentiment_vs_brent_wti.png')
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(complete['date'], complete['sentiment_mean'], color='#2D244D', linewidth=2.5, label='Sentiment (USD)')
ax1.set_ylabel('Sentiment Score\n(Lower is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')

ax2 = ax1.twinx()
ax2.plot(complete['date'], complete['ovx'], color='#FF5B04', linewidth=2.5, label='Oil Volatility Index')
ax2.set_ylabel('Oil Volatility Index\n(Higher is more Volatile)', fontsize=11, fontfamily='monospace')


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Sentiment vs Oil Volatility Index', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()